# Ground Truth Integrity Verification

This notebook verifies the integrity and quality of the ground truth dataset used for model evaluation. It examines class balance, displays representative samples, and conducts data quality checks.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

## Load Dataset and Ground Truth Labels

In [7]:
gt_path = 'auto_labeled_full.csv'
df = pd.read_csv(gt_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names and types:")
print(df.dtypes)
print(f"\nFirst few rows:")
df.head()

Dataset shape: (1522, 12)

Column names and types:
sender_email        object
sender_domain       object
subject             object
clean_text          object
label_source        object
label_topic        float64
label_urgency      float64
pred_decision       object
pred_reasoning      object
pred_confidence      int64
pred_json_valid       bool
pred_latency_ms      int64
dtype: object

First few rows:


,sender_email,sender_domain,subject,clean_text,label_source,label_topic,label_urgency,pred_decision,pred_reasoning,pred_confidence,pred_json_valid,pred_latency_ms
0,onlinecourses@nptel.iitm.ac.in,nptel.iitm.ac.in,NPTEL Newsletter: IIT Madras CODE Certificate Program on Foundations of Machine Learning | Regis...,nptel newsletter: iit madras code certificate program on foundations of machine learning | regis...,External Academic Platform,NaN,NaN,OPPORTUNITY,"The email is an opportunity for students to participate in a certificate program, as indicated b...",100,True,20158
1,support@e.read.ai,e.read.ai,Exclusive Education Pricing Just for You!,"exclusive education pricing just for you! read ai for $5 per month hi 23dcs023 , as a read user ...",External / Misc,NaN,NaN,OPPORTUNITY,"The email offers an exclusive education pricing opportunity, indicating it's optional and not a ...",100,True,7944
2,mohinidarji.dcs@charusat.ac.in,charusat.ac.in,Reg. Submission of SGP- Weekly reports,"reg. submission of sgp- weekly reports dear students, kindly find the attached drive link to upl...",Institutional Sender,NaN,NaN,SUBMIT,"The email requires students to submit their weekly reports, which is an academic work submission.",100,True,8678
3,recommendations@discover.pinterest.com,discover.pinterest.com,Interaction Design for 23DCS023,interaction design for 23dcs023 to view this content open the following url in your browser: <ur...,External / Misc,NaN,NaN,ADMIN,"No actual academic work submission, evaluation, or opportunity; this email appears to be a manda...",100,True,7601
4,sanketsuthar.it@charusat.ac.in,charusat.ac.in,"AWS Academy Batch Starting – 5th July, 10:30 AM @ 628 A Lab","aws academy batch starting – 5th july, 10:30 am @ 628 a lab dear all, this is to inform you that...",Institutional Sender,NaN,NaN,ADMIN,"The email is informing students about a rescheduling of an academy session, which is a mandatory...",100,True,11510


## Class Distribution Statistics

In [8]:
topic_stats = df['pred_decision'].value_counts().sort_index()

print("Topic Distribution:")
print(topic_stats)
print(f"\nTotal topics: {len(topic_stats)}")

print("\n" + "="*60)
print("\nPercentage Distribution by Topic:")
topic_pct = (topic_stats / len(df) * 100).round(2)
for topic, pct in topic_pct.items():
    print(f"  {topic}: {pct}%")

Topic Distribution:
pred_decision
ADMIN          322
EXAM           191
IGNORE         139
OPPORTUNITY    605
SUBMIT         265
Name: count, dtype: int64

Total topics: 5


Percentage Distribution by Topic:
  ADMIN: 21.16%
  EXAM: 12.55%
  IGNORE: 9.13%
  OPPORTUNITY: 39.75%
  SUBMIT: 17.41%


## Sample Verification by Topic (5 samples per class)

In [9]:
def _ascii_clean(text):
    return str(text).encode('ascii', 'ignore').decode('ascii')

for topic in sorted(df['pred_decision'].dropna().unique()):
    print(f"\n{'='*80}")
    print(f"TOPIC: {topic}")
    print(f"{'='*80}")

    topic_df = df[df['pred_decision'] == topic]
    samples = topic_df.head(5)

    for idx, (_, row) in enumerate(samples.iterrows(), 1):
        subject = _ascii_clean(row.get('subject', 'N/A'))
        clean_text = _ascii_clean(row.get('clean_text', 'N/A'))
        print(f"\nSample {idx}:")
        print(f"  Subject: {subject[:100]}")
        print(f"  Text: {clean_text[:150]}..." if len(clean_text) > 150 else f"  Text: {clean_text}")
        print(f"  Label Source: {row.get('label_source', 'N/A')}")


TOPIC: ADMIN

Sample 1:
  Subject: Interaction Design for 23DCS023
  Text: interaction design for 23dcs023 to view this content open the following url in your browser: <url> pinterest 651 brannan street, san francisco, ca, 94...
  Label Source: External / Misc

Sample 2:
  Subject: AWS Academy Batch Starting  5th July, 10:30 AM @ 628 A Lab
  Text: aws academy batch starting  5th july, 10:30 am @ 628 a lab dear all, this is to inform you that the *upcoming saturday is the 4th saturday*, and as pe...
  Label Source: Institutional Sender

Sample 3:
  Subject: AWS Academy Batch Starting  29th June, 10:30 AM @ 628 A Lab
  Text: aws academy batch starting  29th june, 10:30 am @ 628 a lab dear students, we are excited to announce that the aws academy batch will commence on *sat...
  Label Source: Institutional Sender

Sample 4:
  Subject: Internal & External Guide Allocation  5th and 7th Semester Projects
  Text: internal & external guide allocation  5th and 7th semester projects *dear stude

In [12]:
def _ascii_clean(text):
    return str(text).encode('ascii', 'ignore').decode('ascii')

# Extended sample verification - detailed info
for topic in sorted(df['pred_decision'].dropna().unique()):
    print(f"\n{'='*100}")
    print(f"TOPIC: {topic}")
    print(f"{'='*100}")

    topic_df = df[df['pred_decision'] == topic]

    for idx, (_, row) in enumerate(topic_df.iterrows(), 1):
        subject = _ascii_clean(row.get('subject', 'N/A'))
        body = _ascii_clean(row.get('clean_text', 'N/A'))
        print(f"\n[Sample {idx}]")
        print(f"  Source:  {row.get('label_source', 'N/A')}")
        print(f"  Topic:   {row.get('pred_decision', 'N/A')}")
        print(f"  Subject: {subject}")
        print(f"  Body:    {body[:200]}..." if len(body) > 200 else f"  Body:    {body}")
        print(f"  {'-'*96}")


TOPIC: ADMIN

[Sample 1]
  Source:  External / Misc
  Topic:   ADMIN
  Subject: Interaction Design for 23DCS023
  Body:    interaction design for 23dcs023 to view this content open the following url in your browser: <url> pinterest 651 brannan street, san francisco, ca, 94107 help center: <url> privacy policy: <url> terms...
  ------------------------------------------------------------------------------------------------

[Sample 2]
  Source:  Institutional Sender
  Topic:   ADMIN
  Subject: AWS Academy Batch Starting  5th July, 10:30 AM @ 628 A Lab
  Body:    aws academy batch starting  5th july, 10:30 am @ 628 a lab dear all, this is to inform you that the *upcoming saturday is the 4th saturday*, and as per university policy, it will be a holiday. therefo...
  ------------------------------------------------------------------------------------------------

[Sample 3]
  Source:  Institutional Sender
  Topic:   ADMIN
  Subject: AWS Academy Batch Starting  29th June, 10:30 AM @ 628 A 

In [8]:
df=df.drop(['label_topic','label_urgency'],axis=1)

In [9]:
df.to_csv("ground_truth_final.csv")

In [10]:
# Validate taxonomy rules with sample emails
print("="*80)
print("RULE VALIDATION: Sample emails per class")
print("="*80)

keywords_map = {
    "SUBMIT": ["submit", "assignment", "project", "upload", "homework", "sgp", "report"],
    "EXAM": ["exam", "test", "quiz", "marks", "seating", "evaluation", "assessment"],
    "OPPORTUNITY": ["internship", "workshop", "hackathon", "certification", "training", "job"],
    "ADMIN": ["attendance", "marksheet", "enrollment", "fees", "holiday", "timetable", "e-kyc", "official notice"],
    "IGNORE": ["newsletter", "promotion", "event", "notification", "spam"],
}

for label in sorted(df['gt_decision'].unique()):
    label_df = df[df['gt_decision'] == label]
    print(f"\n{label} ({len(label_df)} total)")
    print("-" * 80)
    
    # Show 2 samples
    for idx, (_, row) in enumerate(label_df.head(2).iterrows(), 1):
        subject = str(row.get('subject', 'N/A'))[:60]
        text = str(row.get('clean_text', 'N/A'))[:120]
        
        print(f"\n  Sample {idx}: {subject}")
        
        # Check which keywords match
        combined = (subject + " " + text).lower()
        matched_keywords = set()
        for kw_class, kws in keywords_map.items():
            for kw in kws:
                if kw in combined:
                    matched_keywords.add(f"{kw_class}:{kw}")
        
        if matched_keywords:
            print(f"    Keywords: {', '.join(sorted(matched_keywords)[:5])}")
        else:
            print(f"    Keywords: (none detected)")

print("\n" + "="*80)

RULE VALIDATION: Sample emails per class

ADMIN (28 total)
--------------------------------------------------------------------------------

  Sample 1: Fwd: Regarding extra lectures of MA143 (EM I)
    Keywords: (none detected)

  Sample 2: CE143- CCP Unit tese-1 Syllabus
    Keywords: (none detected)

EXAM (28 total)
--------------------------------------------------------------------------------

  Sample 1: Fwd: Regarding a Quiz-1 of MA253 (DMA) for Semester-3
    Keywords: EXAM:quiz

  Sample 2: Re: Supplementary Examination - 1_ Schedule for B. Tech CSE_
    Keywords: EXAM:exam

IGNORE (28 total)
--------------------------------------------------------------------------------

  Sample 1: HackerRank password reset instructions
    Keywords: (none detected)

  Sample 2: 🐦 Early bird sale: Get Coursera Plus for ₹7,999
    Keywords: (none detected)

OPPORTUNITY (28 total)
--------------------------------------------------------------------------------

  Sample 1: Internship Opportu

## Taxonomy Rules Validation

In [11]:
# Review misclassifications if gt_decision differs from pred_decision
if 'pred_decision' in df.columns:
    mismatches = df[df['pred_decision'] != df['gt_decision']]
    
    if len(mismatches) > 0:
        print("="*80)
        print(f"MISCLASSIFICATIONS: {len(mismatches)} emails mislabeled")
        print("="*80)
        
        for label in sorted(mismatches['gt_decision'].unique()):
            label_mismatches = mismatches[mismatches['gt_decision'] == label]
            print(f"\n{label} → PREDICTED AS:")
            pred_dist = label_mismatches['pred_decision'].value_counts()
            for pred_label, count in pred_dist.items():
                print(f"  → {pred_label}: {count} emails")
            
            print(f"\n  EXAMPLES ({min(3, len(label_mismatches))} shown):")
            for idx, (_, row) in enumerate(label_mismatches.head(3).iterrows(), 1):
                print(f"\n  Example {idx}:")
                print(f"    Subject: {str(row.get('subject', 'N/A'))[:70]}")
                print(f"    Ground Truth: {row.get('gt_decision', 'N/A')}")
                print(f"    Predicted:    {row.get('pred_decision', 'N/A')}")
                print(f"    Reasoning:    {str(row.get('pred_reasoning', 'N/A'))[:100]}")
    else:
        print("✓ Perfect match: All gt_decision labels match pred_decision")

✓ Perfect match: All gt_decision labels match pred_decision


## Sample Review by Class

In [12]:
import pandas as pd
import numpy as np

# Load ground truth
gt_path = 'ground_truth_final.csv'
df = pd.read_csv(gt_path)

print("="*80)
print("GROUND TRUTH INTEGRITY VERIFICATION (5-CLASS TAXONOMY)")
print("="*80)

# Basic stats
print(f"\nDATASET SUMMARY")
print(f"  Total emails: {len(df)}")
print(f"  Columns: {len(df.columns)}")
print(f"  Missing values: {df.isnull().sum().sum()}")

# Check gt_decision column
print(f"\nGT_DECISION COLUMN ANALYSIS")
print(f"  Unique classes: {df['gt_decision'].nunique()}")
print(f"  Classes present: {sorted(df['gt_decision'].unique().tolist())}")

# Class distribution
print(f"\nCLASS DISTRIBUTION (5-CLASS TAXONOMY)")
class_dist = df['gt_decision'].value_counts().sort_index()
print(class_dist)

print(f"\nCLASS DISTRIBUTION (PERCENTAGES)")
class_pct = (class_dist / len(df) * 100).round(2)
for cls, pct in class_pct.items():
    bar = "#" * int(pct / 2)
    print(f"  {cls:12} {pct:6.2f}%  {bar}")

# Check expected 5 classes
expected_classes = {"SUBMIT", "EXAM", "OPPORTUNITY", "ADMIN", "IGNORE"}
actual_classes = set(df['gt_decision'].unique())
missing_classes = expected_classes - actual_classes
extra_classes = actual_classes - expected_classes

print(f"\nTAXONOMY COMPLIANCE")
if missing_classes:
    print(f"  WARNING: Missing classes: {missing_classes}")
else:
    print(f"  OK: All 5 classes present")

if extra_classes:
    print(f"  WARNING: Unexpected classes: {extra_classes}")
else:
    print(f"  OK: No extra classes")

# Check balance
print(f"\nCLASS BALANCE CHECK")
min_class = class_dist.min()
max_class = class_dist.max()
imbalance_ratio = max_class / min_class if min_class > 0 else float('inf')
print(f"  Min samples: {min_class}")
print(f"  Max samples: {max_class}")
print(f"  Imbalance ratio: {imbalance_ratio:.2f}x")
if imbalance_ratio < 2.0:
    print(f"  OK: Good balance (ratio < 2.0)")
elif imbalance_ratio < 3.0:
    print(f"  WARNING: Moderate imbalance (ratio 2-3)")
else:
    print(f"  ERROR: Severe imbalance (ratio > 3)")

# Check for issues
print(f"\nDATA QUALITY ISSUES")
issues = []

if df['gt_decision'].isnull().any():
    issues.append(f"  ERROR: {df['gt_decision'].isnull().sum()} NULL labels in gt_decision")

if (class_dist == 0).any():
    issues.append(f"  ERROR: Some classes have 0 samples")

if 'pred_decision' in df.columns:
    if (df['pred_decision'] != df['gt_decision']).sum() > 0:
        mismatches = (df['pred_decision'] != df['gt_decision']).sum()
        pct_mismatch = (mismatches / len(df) * 100)
        issues.append(f"  WARNING: {mismatches} mismatches between pred_decision and gt_decision ({pct_mismatch:.1f}%)")

if not issues:
    print("  OK: No critical issues detected")
else:
    for issue in issues:
        print(issue)

# Check confidence scores
if 'pred_confidence' in df.columns:
    print(f"\nCONFIDENCE SCORES")
    try:
        conf = pd.to_numeric(df['pred_confidence'], errors='coerce')
        print(f"  Mean: {conf.mean():.3f}")
        print(f"  Min:  {conf.min():.3f}")
        print(f"  Max:  {conf.max():.3f}")
        print(f"  <0.75 (low): {(conf < 0.75).sum()}")
        print(f"  0.75-0.85 (medium): {((conf >= 0.75) & (conf < 0.85)).sum()}")
        print(f"  >=0.85 (high): {(conf >= 0.85).sum()}")
    except:
        print("  (Could not parse confidence scores)")

print("\n" + "="*80)

GROUND TRUTH INTEGRITY VERIFICATION (5-CLASS TAXONOMY)

DATASET SUMMARY
  Total emails: 128
  Columns: 12
  Missing values: 2

GT_DECISION COLUMN ANALYSIS
  Unique classes: 5
  Classes present: ['ADMIN', 'EXAM', 'IGNORE', 'OPPORTUNITY', 'SUBMIT']

CLASS DISTRIBUTION (5-CLASS TAXONOMY)
gt_decision
ADMIN          28
EXAM           28
IGNORE         28
OPPORTUNITY    28
SUBMIT         16
Name: count, dtype: int64

CLASS DISTRIBUTION (PERCENTAGES)
  ADMIN         21.88%  ##########
  EXAM          21.88%  ##########
  IGNORE        21.88%  ##########
  OPPORTUNITY   21.88%  ##########
  SUBMIT        12.50%  ######

TAXONOMY COMPLIANCE
  OK: All 5 classes present
  OK: No extra classes

CLASS BALANCE CHECK
  Min samples: 16
  Max samples: 28
  Imbalance ratio: 1.75x
  OK: Good balance (ratio < 2.0)

DATA QUALITY ISSUES
  OK: No critical issues detected

CONFIDENCE SCORES
  Mean: 97.305
  Min:  0.000
  Max:  100.000
  <0.75 (low): 2
  0.75-0.85 (medium): 0
  >=0.85 (high): 126



## Ground Truth Integrity Check (5-Class Taxonomy)

In [ ]:
# FULL INTEGRITY VERIFICATION
import pandas as pd

gt_path = 'ground_truth_final.csv'
df = pd.read_csv(gt_path, dtype=str, keep_default_na=False)

print("="*80)
print("GROUND TRUTH INTEGRITY VERIFICATION")
print("="*80)

# Overall stats
print(f"\n1. DATASET DIMENSIONS")
print(f"   Total rows: {len(df)}")
print(f"   Total columns: {len(df.columns)}")

# Check for empty/null values
print(f"\n2. EMPTY/NULL COLUMNS")
for col in df.columns:
    empty_count = (df[col] == '').sum() + df[col].isna().sum()
    if empty_count > 0:
        print(f"   {col}: {empty_count} empty values")

# Class distribution - GT labels
print(f"\n3. CLASS DISTRIBUTION (gt_decision)")
if 'gt_decision' in df.columns:
    dist = df['gt_decision'].value_counts().sort_index()
    print(f"   {dist.to_dict()}")
    print(f"   Total classes: {df['gt_decision'].nunique()}")
    print(f"   Classes present: {sorted(df['gt_decision'].unique().tolist())}")

# Check alignment between pred and gt
print(f"\n4. PREDICTION vs GROUND TRUTH ALIGNMENT")
if 'pred_decision' in df.columns and 'gt_decision' in df.columns:
    match_count = (df['pred_decision'] == df['gt_decision']).sum()
    print(f"   Matches: {match_count}/{len(df)} ({100*match_count/len(df):.1f}%)")

    mismatches = df[df['pred_decision'] != df['gt_decision']]
    if len(mismatches) > 0:
        print(f"\n   Mismatches found:")
        for idx, row in mismatches.head(10).iterrows():
            print(f"      Row {idx}: pred={row['pred_decision']}, gt={row['gt_decision']}")

# Confidence scores
print(f"\n5. CONFIDENCE SCORES")
if 'pred_confidence' in df.columns:
    con_numeric = pd.to_numeric(df['pred_confidence'], errors='coerce')
    print(f"   Mean: {con_numeric.mean():.2f}")
    print(f"   Min: {con_numeric.min():.0f}")
    print(f"   Max: {con_numeric.max():.0f}")
    print(f"   <75% confidence: {(con_numeric < 75).sum()}")

# JSON validity
print(f"\n6. JSON PARSING SUCCESS")
if 'pred_json_valid' in df.columns:
    valid_count = (df['pred_json_valid'] == 'True').sum()
    print(f"   Valid JSON: {valid_count}/{len(df)} ({100*valid_count/len(df):.1f}%)")

# Data quality
print(f"\n7. TEXT FIELD QUALITY")
print(f"   Subject field - non-empty: {(df['subject'] != '').sum()}")
print(f"   Clean text field - non-empty: {(df['clean_text'] != '').sum()}")
print(f"   Label source non-empty: {(df['label_source'] != '').sum()}")

# Potential issues
print(f"\n8. POTENTIAL ISSUES")
issues = 0

if df['gt_decision'].nunique() < 5:
    print(f"   ERROR: Class distribution incomplete: {df['gt_decision'].nunique()}/5 classes")
    print(f"      Missing: {set(['SUBMIT', 'EXAM', 'OPPORTUNITY', 'ADMIN', 'IGNORE']) - set(df['gt_decision'].unique())}")
    issues += 1

# Check balance
class_counts = df['gt_decision'].value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()
if imbalance_ratio > 2:
    print(f"   WARNING: Class imbalance detected: {imbalance_ratio:.1f}x ratio")
    print(f"      Max: {class_counts.max()}, Min: {class_counts.min()}")
    issues += 1

if (df['pred_confidence'].astype(float) < 75).sum() > len(df) * 0.1:
    print(f"   WARNING: Low confidence predictions: {(df['pred_confidence'].astype(float) < 75).sum()}")
    issues += 1

if issues == 0:
    print(f"   OK: No critical issues detected")

print(f"\n" + "="*80)